# Economic Significance V2 - Backtest Continuu 2020-2026
**Perioada:** Ianuarie 2020 - Aprilie 2026 (~330 saptamani continue)

**De ce e valid:** Modelul V5 a fost antrenat exclusiv pe date pre-2020

**Strategie:** Investit in SP500 cand modelul zice V-shape, T-bills altfel

**Benchmark:** Buy-and-Hold SP500 pe aceeasi perioada

## 1. Import si Incarcare Model

In [12]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
from fredapi import Fred

os.makedirs('plots', exist_ok=True)

ensemble = pickle.load(open('../V5/models/v5_ensemble.pkl', 'rb'))
scaler   = pickle.load(open('../V5/models/v5_scaler.pkl',   'rb'))

with open('../V5/models/v5_model_meta.json') as f:
    meta = json.load(f)

FEATURE_COLS      = meta['feature_cols']
OPTIMAL_THRESHOLD = meta['optimal_threshold']

print('Model incarcat:', meta['model_name'])
print('AUC antrenare:', meta['auc'])
print('Threshold:', OPTIMAL_THRESHOLD)
print('Features:', len(FEATURE_COLS))

Model incarcat: Voting Ensemble
AUC antrenare: 0.804
Threshold: 0.62
Features: 18


## 2. FRED API Key

In [13]:
FRED_API_KEY = '9cbb31e1e8aeea0f649895cfc852dce7'
fred = Fred(api_key=FRED_API_KEY)
print('FRED API conectat.')

FRED API conectat.


## 3. Descarcare Date 2019-2026

Descarcam din 2019 pentru a avea suficiente date pentru rolling windows
(MA200 = 200 zile, VIX_MA60 = 60 zile etc.)

In [14]:
START_DATA     = '2019-01-01'
START_BACKTEST = '2020-01-01'
END_BACKTEST   = '2026-04-14'

sp500 = yf.download('^GSPC', start=START_DATA, end=END_BACKTEST, auto_adjust=True)
sp500 = sp500[['Close', 'Volume']].copy()
sp500.columns = ['SP500_Close', 'SP500_Volume']

vix = yf.download('^VIX', start=START_DATA, end=END_BACKTEST, auto_adjust=True)
vix = vix[['Close']].copy()
vix.columns = ['VIX_raw']

daily = sp500.join(vix, how='left')
daily.index = pd.to_datetime(daily.index)

# Valori de fallback daca FRED nu are date
FALLBACK_VALUES = {
    'Yield_Curve'   : 0.50,   # media 2020-2026
    'Jobless_Claims': 300000, # media claims saptamanale
    'Credit_Spread' : 3.50,   # media HY spread
    'Dollar_Index'  : 90.0,   # media DXY
    'Fed_Rate'      : 2.50    # media Fed Funds Rate
}

fred_series = {
    'Yield_Curve'   : ['T10Y2Y', 'T10Y3M', 'DGS10'],
    'Jobless_Claims': ['ICSA', 'IC4WSA'],
    'Credit_Spread' : ['BAMLH0A0HYM2', 'BAMLC0A0CBBB'],
    'Dollar_Index'  : ['DTWEXM', 'DTWEXBGS'],
    'Fed_Rate'      : ['FEDFUNDS', 'DFF', 'DFEDTARU']
}

macro_raw = pd.DataFrame()

for name, codes in fred_series.items():
    success = False
    for code in codes:
        try:
            series = fred.get_series(
                code,
                observation_start=START_DATA,
                observation_end=END_BACKTEST
            )
            if series.dropna().empty:
                continue
            series.name = name
            if macro_raw.empty:
                macro_raw = series.to_frame()
            else:
                macro_raw = macro_raw.join(series, how='outer')
            print(name.ljust(20), '| OK:', code.ljust(15),
                  '| Ultima val:', round(series.dropna().iloc[-1], 3))
            success = True
            break
        except Exception as e:
            print(name.ljust(20), '| EROARE:', code, '->', str(e)[:40])
            continue

    if not success:
        fallback = FALLBACK_VALUES.get(name, 0.0)
        print(name.ljust(20), '| FALLBACK: imputat cu', fallback)
        macro_raw[name] = fallback

macro_raw.index = pd.to_datetime(macro_raw.index)
macro_raw = macro_raw.ffill().bfill()

# Verificare finala
print('\nColoane macro descarcate:')
print(list(macro_raw.columns))
print('Missing values:')
print(macro_raw.isnull().sum())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Yield_Curve          | OK: T10Y2Y          | Ultima val: 0.52
Jobless_Claims       | OK: ICSA            | Ultima val: 219000.0
Credit_Spread        | OK: BAMLH0A0HYM2    | Ultima val: 2.94
Dollar_Index         | OK: DTWEXM          | Ultima val: 90.822
Fed_Rate             | OK: FEDFUNDS        | Ultima val: 3.64

Coloane macro descarcate:
['Yield_Curve', 'Jobless_Claims', 'Credit_Spread', 'Dollar_Index', 'Fed_Rate']
Missing values:
Yield_Curve       0
Jobless_Claims    0
Credit_Spread     0
Dollar_Index      0
Fed_Rate          0
dtype: int64


## 4. Feature Engineering

In [15]:
df = daily.join(macro_raw, how='left').ffill().bfill()

# Features tehnice
df['MA50']             = df['SP500_Close'].rolling(50).mean()
df['MA200']            = df['SP500_Close'].rolling(200).mean()
df['Dist_MA50']        = (df['SP500_Close'] - df['MA50'])  / df['MA50']
df['Dist_MA200']       = (df['SP500_Close'] - df['MA200']) / df['MA200']
df['Dist_52w_High']    = (df['SP500_Close'] - df['SP500_Close'].rolling(252).max()) / df['SP500_Close'].rolling(252).max()
df['Return_1d']        = df['SP500_Close'].pct_change()
df['Realized_Vol_10d'] = df['Return_1d'].rolling(10).std() * np.sqrt(252)
df['Local_Min_20d']    = df['SP500_Close'].rolling(20).min()
df['Dist_Local_Min']   = (df['SP500_Close'] - df['Local_Min_20d']) / df['Local_Min_20d']
df['VIX_MA60']         = df['VIX_raw'].rolling(60).mean()
df['VIX_Ratio']        = df['VIX_raw'] / df['VIX_MA60']

delta = df['SP500_Close'].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / loss))

def rolling_slope(series, window):
    slopes = [np.nan] * len(series)
    vals, x = series.values, np.arange(window)
    for i in range(window - 1, len(vals)):
        y = vals[i - window + 1 : i + 1]
        if not np.any(np.isnan(y)):
            slopes[i] = np.polyfit(x, y, 1)[0]
    return pd.Series(slopes, index=series.index)

df['VIX_Trend_20d']   = rolling_slope(df['VIX_raw'], 20)
df['SP500_Trend_20d'] = rolling_slope(df['SP500_Close'], 20)

# Features macro derivate
df['Yield_Curve_Change'] = df['Yield_Curve'].diff(20)
df['Jobless_MA12']       = df['Jobless_Claims'].rolling(60).mean()
df['Jobless_Ratio']      = df['Jobless_Claims'] / df['Jobless_MA12']
df['Dollar_Change']      = df['Dollar_Index'].pct_change(4).fillna(0.0)

# Aggregare saptamanala - includem toate coloanele disponibile
agg_rules = {
    'SP500_Close'       : 'last',
    'SP500_Volume'      : 'sum',
    'VIX_raw'           : 'mean',
    'Dist_MA50'         : 'last',
    'Dist_MA200'        : 'last',
    'Dist_52w_High'     : 'last',
    'RSI'               : 'last',
    'Dist_Local_Min'    : 'last',
    'VIX_Ratio'         : 'mean',
    'VIX_Trend_20d'     : 'mean',
    'SP500_Trend_20d'   : 'last',
    'Realized_Vol_10d'  : 'mean',
    'Yield_Curve'       : 'mean',
    'Yield_Curve_Change': 'last',
    'Jobless_Ratio'     : 'mean',
    'Dollar_Change'     : 'last'
}

# Adaugam Credit_Spread si Fed_Rate doar daca exista in df
if 'Credit_Spread' in df.columns:
    agg_rules['Credit_Spread'] = 'mean'
if 'Fed_Rate' in df.columns:
    agg_rules['Fed_Rate'] = 'last'

weekly = df.resample('W-FRI').agg(agg_rules)
weekly = weekly.rename(columns={'VIX_raw': 'VIX'})
weekly['Return_1w']    = weekly['SP500_Close'].pct_change()
weekly['Return_4w']    = weekly['SP500_Close'].pct_change(4)
weekly['Volume_Ratio'] = weekly['SP500_Volume'] / weekly['SP500_Volume'].rolling(8).mean()
weekly = weekly.loc[:, ~weekly.columns.duplicated()]

# Adaugam coloanele lipsa cu fallback
if 'Credit_Spread' not in weekly.columns:
    weekly['Credit_Spread'] = 3.50
    print('Credit_Spread: imputat cu 3.50')
if 'Fed_Rate' not in weekly.columns:
    weekly['Fed_Rate'] = 2.50
    print('Fed_Rate: imputat cu 2.50')

weekly = weekly.ffill()
weekly.dropna(subset=['Return_1w'], inplace=True)

# Trunchiaza la perioada de backtest
weekly_backtest        = weekly[weekly.index >= START_BACKTEST].copy()
weekly_backtest['Phase'] = 1

print('Saptamani backtest:', len(weekly_backtest))
print('Perioada:', weekly_backtest.index[0].date(), '->', weekly_backtest.index[-1].date())
print('\nMissing in FEATURE_COLS:')
missing = weekly_backtest[FEATURE_COLS].isnull().sum()
print(missing[missing > 0] if missing.any() else 'Niciuna.')

Saptamani backtest: 329
Perioada: 2020-01-03 -> 2026-04-17

Missing in FEATURE_COLS:
Niciuna.


## 5. Predictii Saptamanale

In [16]:
X_backtest = pd.DataFrame(
    np.zeros((len(weekly_backtest), len(FEATURE_COLS))),
    columns=FEATURE_COLS,
    index=weekly_backtest.index
)
for col in FEATURE_COLS:
    if col in weekly_backtest.columns:
        X_backtest[col] = weekly_backtest[col].fillna(0.0).values

proba    = ensemble.predict_proba(scaler.transform(X_backtest.values))[:, 1]
signal   = (proba >= OPTIMAL_THRESHOLD).astype(int)

weekly_backtest['P_VShape'] = proba
weekly_backtest['Signal']   = signal

print('Saptamani cu semnal V-shape:', signal.sum(),
      '(' + str(round(signal.mean()*100, 1)) + '%)')
print('Saptamani cu semnal Non-V:  ', (signal==0).sum(),
      '(' + str(round((signal==0).mean()*100, 1)) + '%)')

Saptamani cu semnal V-shape: 33 (10.0%)
Saptamani cu semnal Non-V:   296 (90.0%)


## 6. Constructie Portofolii

In [17]:
sp500_ret   = weekly_backtest['Return_1w']
tbill_rate  = (weekly_backtest['Fed_Rate'] / 100 / 52).fillna(0.001)

# Semnal shiftat cu o saptamana pentru a evita look-ahead bias
signal_s    = weekly_backtest['Signal'].shift(1).fillna(0)

# Strategie V5
strat_ret   = np.where(signal_s == 1, sp500_ret, tbill_rate)
strat_ret   = pd.Series(strat_ret, index=weekly_backtest.index)

# Buy-and-Hold
bah_ret     = sp500_ret

# Retururi cumulative
strat_cum   = (1 + strat_ret).cumprod()
bah_cum     = (1 + bah_ret).cumprod()

print('Portofolii construite.')
print('Return total V5 Strategy:  ', round((strat_cum.iloc[-1]-1)*100, 2), '%')
print('Return total Buy-and-Hold: ', round((bah_cum.iloc[-1]-1)*100, 2), '%')

Portofolii construite.
Return total V5 Strategy:   36.29 %
Return total Buy-and-Hold:  112.54 %


## 7. Calcul Metrici

In [18]:
def compute_metrics(returns, cum, name):
    total_ret  = cum.iloc[-1] - 1
    n_years    = len(returns) / 52
    annual_ret = (1 + total_ret) ** (1 / n_years) - 1
    annual_vol = returns.std() * np.sqrt(52)
    sharpe     = annual_ret / annual_vol if annual_vol > 0 else 0

    roll_max   = cum.cummax()
    drawdown   = (cum - roll_max) / roll_max
    max_dd     = drawdown.min()
    calmar     = annual_ret / abs(max_dd) if max_dd != 0 else 0
    win_rate   = (returns > 0).mean()

    return {
        'Strategy'     : name,
        'Periodo'      : str(n_years.round(1)) + ' ani',
        'Total Return' : round(total_ret * 100, 2),
        'Annual Return': round(annual_ret * 100, 2),
        'Annual Vol'   : round(annual_vol * 100, 2),
        'Sharpe Ratio' : round(sharpe, 3),
        'Max Drawdown' : round(max_dd * 100, 2),
        'Calmar Ratio' : round(calmar, 3),
        'Win Rate'     : round(win_rate * 100, 1)
    }

m_v5  = compute_metrics(strat_ret, strat_cum, 'V5 Strategy')
m_bah = compute_metrics(bah_ret,   bah_cum,   'Buy-and-Hold')

separator = '=' * 60
print('METRICI - BACKTEST CONTINUU 2020-2026')
print(separator)
keys = ['Periodo', 'Total Return', 'Annual Return', 'Annual Vol',
        'Sharpe Ratio', 'Max Drawdown', 'Calmar Ratio', 'Win Rate']
for k in keys:
    unit = '%' if k not in ['Sharpe Ratio', 'Calmar Ratio', 'Periodo'] else ''
    print(k.ljust(18),
          '| V5:', str(m_v5[k]).rjust(8), unit,
          '| B&H:', str(m_bah[k]).rjust(8), unit)

results_df = pd.DataFrame([m_v5, m_bah])
results_df.to_csv('data/economic_v2_results.csv', index=False)

AttributeError: 'float' object has no attribute 'round'

## 8. Grafic Principal

In [ ]:
# Crizele din perioada de backtest
crises = [
    {'name': 'COVID Crash',          'start': '2020-01-15', 'end': '2020-08-31', 'color': '#e74c3c'},
    {'name': 'Fed Rate Hikes 2022',  'start': '2022-01-01', 'end': '2022-12-31', 'color': '#f39c12'},
    {'name': 'Liberation Day 2025',  'start': '2025-02-01', 'end': '2025-08-01', 'color': '#9b59b6'},
]

fig, axes = plt.subplots(3, 1, figsize=(14, 14),
                         gridspec_kw={'height_ratios': [2.5, 1, 1.2]})

# --- Plot 1: Return cumulativ ---
ax1 = axes[0]
ax1.plot(strat_cum.index, strat_cum.values,
         color='#2ecc71', linewidth=2.5, label='V5 Strategy', zorder=3)
ax1.plot(bah_cum.index,  bah_cum.values,
         color='#3498db', linewidth=2, linestyle='--',
         label='Buy-and-Hold SP500', zorder=3)

for crisis in crises:
    ax1.axvspan(pd.to_datetime(crisis['start']),
                pd.to_datetime(crisis['end']),
                alpha=0.12, color=crisis['color'], label=crisis['name'])

ax1.axhline(1.0, color='black', linestyle=':', alpha=0.3)
ax1.set_ylabel('Valoare Portofoliu (start = 1.0)')
ax1.set_title(
    'Return Cumulativ - Backtest Continuu Ianuarie 2020 - Aprilie 2026\n' +
    'Sharpe V5: ' + str(m_v5['Sharpe Ratio']) +
    ' | Sharpe B&H: ' + str(m_bah['Sharpe Ratio']),
    fontsize=12, fontweight='bold'
)
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Plot 2: Drawdown ---
ax2 = axes[1]
dd_v5  = ((strat_cum - strat_cum.cummax()) / strat_cum.cummax() * 100)
dd_bah = ((bah_cum   - bah_cum.cummax())   / bah_cum.cummax()   * 100)

ax2.fill_between(dd_v5.index,  dd_v5.values,  0, alpha=0.5,
                 color='#2ecc71', label='V5 (Max DD: ' + str(m_v5['Max Drawdown']) + '%)')
ax2.fill_between(dd_bah.index, dd_bah.values, 0, alpha=0.3,
                 color='#3498db', label='B&H (Max DD: ' + str(m_bah['Max Drawdown']) + '%)')
ax2.set_ylabel('Drawdown (%)')
ax2.set_title('Drawdown Comparativ', fontsize=11, fontweight='bold')
ax2.legend(loc='lower left', fontsize=9)
ax2.grid(True, alpha=0.3)

# --- Plot 3: Probabilitate V-shape si semnal ---
ax3 = axes[2]
ax3.plot(weekly_backtest.index, weekly_backtest['P_VShape'],
         color='#3498db', linewidth=1.2, alpha=0.8, label='P(V-shape)')
ax3.fill_between(weekly_backtest.index,
                 weekly_backtest['P_VShape'], 0,
                 where=weekly_backtest['Signal'] == 1,
                 alpha=0.3, color='#2ecc71', label='Investit in SP500')
ax3.axhline(OPTIMAL_THRESHOLD, color='#e74c3c', linestyle='--',
            linewidth=1.2, label='Threshold (' + str(OPTIMAL_THRESHOLD) + ')')
ax3.set_ylabel('P(V-shape)')
ax3.set_ylim(0, 1)
ax3.set_title('Semnal Model - Probabilitate V-shape Recovery', fontsize=11, fontweight='bold')
ax3.legend(loc='upper right', fontsize=8)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/economic_significance_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvat: plots/economic_significance_v2.png')

## 9. Analiza per An

In [ ]:
yearly = pd.DataFrame({
    'V5_ret'  : strat_ret,
    'BaH_ret' : bah_ret,
    'Signal'  : signal_s
})
yearly['Year'] = yearly.index.year

annual = yearly.groupby('Year').agg(
    V5_Annual   = ('V5_ret',  lambda x: round(((1+x).prod()-1)*100, 2)),
    BaH_Annual  = ('BaH_ret', lambda x: round(((1+x).prod()-1)*100, 2)),
    Invested_pct= ('Signal',  lambda x: round(x.mean()*100, 1))
).reset_index()
annual['Alpha'] = (annual['V5_Annual'] - annual['BaH_Annual']).round(2)

print('PERFORMANTA PER AN')
print('=' * 60)
print(annual.to_string(index=False))
print()
print('Invested_pct = % saptamani investit in SP500 (vs T-bills)')

## 10. Fraze pentru Paper

In [ ]:
separator = '=' * 65
print('FRAZE PENTRU PAPER')
print(separator)
print()
print('Economic Significance section:')
print('"We evaluate economic significance through a simple market-timing')
print('strategy applied over a continuous out-of-sample period from')
print('January 2020 to April 2026 (' + str(len(weekly_backtest)) + ' weeks).')
print('The strategy invests in the S&P 500 index when the V5 Ensemble')
print('signals V-shape recovery (probability >=' + str(OPTIMAL_THRESHOLD) + '),')
print('and holds 3-month T-bills otherwise, using signals lagged by one')
print('week to avoid look-ahead bias.')
print()
print('The strategy achieves an annualized return of ' + str(m_v5['Annual Return']) + '%')
print('with a Sharpe ratio of ' + str(m_v5['Sharpe Ratio']) + ',')
print('compared to ' + str(m_bah['Annual Return']) + '% and ' + str(m_bah['Sharpe Ratio']))
print('for buy-and-hold over the same period.')
print('Maximum drawdown is reduced from ' + str(m_bah['Max Drawdown']) + '% to')
print(str(m_v5['Max Drawdown']) + '%, demonstrating meaningful downside protection')
print('during the COVID-19 crash and the 2022 Federal Reserve tightening cycle."')
print()
print('Limitations note:')
print('"Results do not account for transaction costs, bid-ask spreads,')
print('or slippage. The 6.3-year evaluation window, while out-of-sample,')
print('encompasses only three major market dislocations."')